# 04. Embedding Model Benchmarking

This notebook evaluates various embedding models for their throughput, latency, and dimensionality. A robust embedding model is foundational to vector-based semantic retrieval.

In [ ]:
import sys
import os
import json
import matplotlib.pyplot as plt

# Add repository root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from colab.src import utils
from colab.src import dataset_utils
from colab.src import embeddings

# Setup environment
utils.set_seed(42)
config = utils.load_config('experiment_config.yaml')
utils.print_header("Embedding Benchmark Setup Complete")

## 1. Load Dataset Subset
We'll use a small controlled subset (500 passages) to benchmark inference times.

In [ ]:
dataset = dataset_utils.load_msmarco_xi(config)
all_passages = dataset_utils.extract_all_passages(dataset)
benchmark_passages = all_passages[:500]
print(f"Using {len(benchmark_passages)} passages for benchmarking.")

## 2. Selected Models

We will evaluate the following models:

1. **intfloat/multilingual-e5-small**: Lightweight, fast, good multilingual performance.
2. **intfloat/multilingual-e5-base**: Stronger performance, but heavier.
3. **sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2**: Standard fast baseline for dense retrieval.
4. **BAAI/bge-m3**: Leading multilingual dense model with dense/sparse capability support.

In [ ]:
EMBEDDING_MODELS = [
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "BAAI/bge-m3",
]

## 3. Run Benchmark
Measuring throughput, latency, dimensionality, and model size.

In [ ]:
print("Starting embedding benchmark...")
results = embeddings.benchmark_models(EMBEDDING_MODELS, benchmark_passages)
utils.print_table(results, title="Embedding Model Benchmark Results")

## 4. Evaluation and Normalization

### Normalization Strategy
Cosine similarity measures the angle between vectors. When vectors are L2-normalized, the **Inner Product** becomes mathematically equivalent to Cosine Similarity. 
Using normalized vectors with Inner Product indices (like FAISS `IndexFlatIP`) avoids computing costly square roots during query-time, yielding substantially faster retrieval.

In [ ]:
# Visualization of Latency vs Model Size
model_names = [res['model'] for res in results]
latencies = [res['avg_latency_ms'] for res in results]
throughputs = [res['throughput'] for res in results]

fig, ax1 = plt.subplots(figsize=(12, 6))

ax2 = ax1.twinx()
ax1.bar(model_names, latencies, color='skyblue', label='Avg Latency (ms)')
ax2.plot(model_names, throughputs, color='darkred', marker='o', label='Throughput (passages/s)')

ax1.set_xlabel('Model')
ax1.set_ylabel('Avg Latency (ms)', color='skyblue')
ax2.set_ylabel('Throughput (passages/s)', color='darkred')
plt.title('Embedding Models: Latency vs Throughput')
ax1.set_xticklabels(model_names, rotation=45, ha='right')
fig.tight_layout()
plt.show()

## 5. Save Results

In [ ]:
reports_dir = utils.get_reports_dir()
report_path = os.path.join(reports_dir, 'embedding_benchmark.json')
utils.save_json(results, report_path)
print(f"Results saved to {report_path}")

## Decision
When choosing a model, we balance retrieval quality (recall) against inference latency.

- **MiniLM** provides high throughput and low latency, making it an excellent baseline.
- **BGE-M3** offers exceptional retrieval capabilities but may increase query latency.
- **E5-small** is a great compromise for multilingual workloads.

*The optimal model will be configured in `experiment_config.yaml` for the end-to-end RAG pipeline based on our latency budget.*